# 高麗菜估價（老師版）

老師對照用，不發給學生。格子跟 `高麗菜估價.ipynb` 一樣：1 資料、2 前處理、3 訓練、4 你的判斷。

在 repo 的 `linear-regression/teacher/` 直接打開就能跑，放進學生專案的 `線性回歸/` 實驗夾也能跑。只用 numpy、matplotlib，寫法跟 `線性回歸.ipynb` 一樣：`loadData`、z-score、Cost、梯度下降都自己寫。

模型：

$$
y = w_1 \ln(\text{交易量}) + w_2 x_{2025} + w_3 x_{2026} + b
$$

$x_{2025}$、$x_{2026}$ 是 0/1：那天是 2025 年就 $x_{2025}=1$，是 2026 年就 $x_{2026}=1$。2024 年兩個都是 0，是比較的基準。

| | 參考值（全部 796 天） | 這本（隨機 80% 訓練） |
|---|---|---|
| $w_1$（ln 交易量） | −15.89 | 約 −15.88 |
| $w_2$（2025 年） | −10.14 | 約 −9.57 |
| $w_3$（2026 年） | −7.95 | 約 −7.91 |
| $b$ | 87.66 | 約 87.31 |
| 只用 ln(交易量) | $w=-15.38$，$b=80.05$ | 約 $-15.42$、$80.12$ |

隨機切 20% 當測試時，係數會在參考值附近晃動，大約 ±0.6 以內。


## 1 資料


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

def loadData(filename, dtype=float):
    try:
        here = Path(__vsc_ipynb_file__).resolve().parent
    except NameError:
        here = Path.cwd()
    # 放在學生的 線性回歸/ 實驗夾：資料在旁邊的 data/
    # 留在 repo 的 linear-regression/teacher/：資料在套件的 experiment/data/
    folders = [here / "data", here.parent / "src" / "add_linear_regression" / "experiment" / "data"]
    folder = next((f for f in folders if (f / filename).is_file()), folders[0])
    return np.loadtxt(
        folder / filename,
        delimiter=",",
        skiprows=1,
        encoding="utf-8",
        dtype=dtype,
    )

# 日期是文字、天氣有空白，先全部當文字讀進來，要用的欄再轉成數字
rows = loadData("taipei2_cabbage.csv", dtype=str)
columns = ["日期", "月份", "平均批發價", "交易量", "近30日均溫", "近7日雨量"]
print("筆數", len(rows))
for j, name in enumerate(columns):
    print(f"{name} 空白 {(rows[:, j] == '').sum()} 格")

dates = rows[:, 0]
price = rows[:, 2].astype(float)    # 平均批發價（元/公斤），要預測的
volume = rows[:, 3].astype(float)   # 交易量（噸）
year = np.array([int(d[:4]) for d in dates])
print("交易量 最小", volume.min(), "最大", volume.max())
print("各年天數", {int(y): int((year == y).sum()) for y in np.unique(year)})

# 交易量少時價格衝很高，交易量多時價格慢慢壓低，不是直線
plt.scatter(volume, price, s=8, alpha=0.4)
plt.xlabel("Volume (tonnes)")
plt.ylabel("Average price (NTD/kg)")
plt.title("Price vs. volume")
plt.show()


## 2 前處理

不用的欄：

- **氣溫、雨量**：有空白。先不放，在「你的判斷」再檢查放進去有沒有差。
- **月份**：這本沒放，同樣在「你的判斷」檢查。

用到的只有交易量和日期，兩欄都沒有空白，796 天全部留下。


In [ ]:
# 特徵 1：交易量取 ln，把「少量時價格衝高、多量時慢慢平」拉成接近直線
# 特徵 2、3：年份。2024 年是基準，2025、2026 各用一欄 0/1 表示
X_all = np.column_stack([
    np.log(volume),
    (year == 2025).astype(float),
    (year == 2026).astype(float),
])
y_all = price
print("前處理後", X_all.shape)

# 打亂後 80% 訓練、20% 測試
# seed 11 是典型的一種切法：測試 Cost 跟訓練 Cost 差不多，係數也接近參考值
rng = np.random.default_rng(11)
order = rng.permutation(len(X_all))
cut = int(len(X_all) * 0.8)
train, test = order[:cut], order[cut:]
X_train, y_train = X_all[train], y_all[train]
X_test, y_test = X_all[test], y_all[test]
print("訓練", X_train.shape, "測試", X_test.shape)

# ln(交易量) 大約在 -0.5 到 4.8，年份是 0/1，尺度不同，用 z-score 縮放
# 平均和標準差只用訓練資料算，測試資料套用同一組
def zscore_normalize_features(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    return (X - mu) / sigma, mu, sigma

X_train_norm, mu, sigma = zscore_normalize_features(X_train)
X_test_norm = (X_test - mu) / sigma
print("訓練資料的 mu", mu.round(3), "sigma", sigma.round(3))


## 3 訓練


In [ ]:
def compute_cost_multi(X, y, w, b):
    m = X.shape[0]
    prediction = np.dot(X, w) + b
    return np.sum((prediction - y) ** 2) / (2 * m)

def compute_gradient_multi(X, y, w, b):
    m = X.shape[0]
    errors = np.dot(X, w) + b - y
    dj_dw = np.dot(X.T, errors) / m
    dj_db = np.sum(errors) / m
    return dj_dw, dj_db

def gradient_descent_multi(X, y, alpha, num_iters):
    w = np.zeros(X.shape[1])
    b = 0.0
    history = []
    for _ in range(num_iters):
        dj_dw, dj_db = compute_gradient_multi(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        history.append(compute_cost_multi(X, y, w, b))
    return w, b, history

def to_original_scale(w, b, mu, sigma):
    # 縮放後學到的 w、b 換回原始單位，才能跟參考值比，也比較好解釋
    return w / sigma, b - np.sum(w * mu / sigma)

w_norm, b_norm, history = gradient_descent_multi(X_train_norm, y_train, alpha=0.1, num_iters=1000)
w, b = to_original_scale(w_norm, b_norm, mu, sigma)
print("換回原始單位的 w =", w.round(2), "b =", round(float(b), 2))
print("參考值約 w=[-15.89, -10.14, -7.95], b=87.66（用全部資料算的，這裡只用 80%，會差一點）")

cost_train = compute_cost_multi(X_train_norm, y_train, w_norm, b_norm)
cost_test = compute_cost_multi(X_test_norm, y_test, w_norm, b_norm)
# Cost = 平均平方誤差 / 2，所以 sqrt(2 * Cost) 是「平均大約差幾元」
print(f"訓練 Cost {cost_train:.2f}（約差 {np.sqrt(2 * cost_train):.1f} 元/公斤）")
print(f"測試 Cost {cost_test:.2f}（約差 {np.sqrt(2 * cost_test):.1f} 元/公斤）")

plt.plot(history)
plt.xlabel("epoch")
plt.ylabel("Cost")
plt.title("Cost should fall, then flatten")
plt.show()


In [ ]:
# 對照：只用 ln(交易量)
w_one_norm, b_one_norm, _ = gradient_descent_multi(X_train_norm[:, :1], y_train, alpha=0.1, num_iters=1000)
w_one, b_one = to_original_scale(w_one_norm, b_one_norm, mu[:1], sigma[:1])
cost_one_test = compute_cost_multi(X_test_norm[:, :1], y_test, w_one_norm, b_one_norm)
print("只用 ln(交易量)：w =", w_one.round(2), "b =", round(float(b_one), 2), "（參考值約 -15.38、80.05）")
print(f"只用 ln(交易量) 的測試 Cost {cost_one_test:.2f}，加上年份 {cost_test:.2f}")


In [ ]:
# 貼合程度：測試資料的點，加上三個年份各自的曲線
v_line = np.linspace(volume.min(), volume.max(), 200)
colors = {2024: "tab:blue", 2025: "tab:orange", 2026: "tab:green"}
for yr, color in colors.items():
    mask = year[test] == yr
    plt.scatter(np.exp(X_test[mask, 0]), y_test[mask], s=12, c=color, alpha=0.6, label=f"{yr} test days")
    curve = w[0] * np.log(v_line) + w[1] * (yr == 2025) + w[2] * (yr == 2026) + b
    plt.plot(v_line, curve, c=color)
plt.xlabel("Volume (tonnes)")
plt.ylabel("Average price (NTD/kg)")
plt.title("Test days and fitted curves by year")
plt.legend()
plt.show()

# 測試資料的實際價格對預測價格，點貼著虛線表示估得準
pred_test = np.dot(X_test_norm, w_norm) + b_norm
lims = [min(y_test.min(), pred_test.min()), max(y_test.max(), pred_test.max())]
plt.scatter(y_test, pred_test, s=12, alpha=0.6)
plt.plot(lims, lims, "k--", label="perfect")
plt.xlabel("Actual price (NTD/kg)")
plt.ylabel("Predicted price (NTD/kg)")
plt.title("Test days: actual vs. predicted")
plt.legend()
plt.show()


<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">4 你的判斷（參考答案，學生要自己寫）</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;"><b>交易量變多時，價格怎麼走？</b><br>
w<sub>1</sub> 約 −16：交易量變成 2 倍，價格大約少 16 × ln2 ≈ 11 元/公斤。同樣多 10 噸，交易量少的時候價格掉很多，交易量多的時候只掉一點，所以用 ln 比直線貼。</li>
<li style="color:#1a1a1a;"><b>年份</b><br>
同樣的交易量，2025 年比 2024 年便宜約 10 元，2026 年便宜約 8 元。這是已經發生的年份的行情水準，只能拿來解釋，不能拿來估還沒發生的 2027 年。</li>
<li style="color:#1a1a1a;"><b>溫度或雨量有沒有幫助到預測？或是影響你的判斷？</b>（下一格各加一次看看）<br>
<b>雨量</b>：測試 Cost 和 w<sub>1</sub> 都幾乎不變，沒有影響。<br>
<b>溫度</b>：測試 Cost 只差一點，但 w<sub>1</sub> 從約 −16 變成約 −13。溫度和 ln(交易量) 的相關係數約 −0.74，天熱時交易量本來就少，兩欄講的是重疊的事，「交易量的影響」被溫度分走一部分。方向不變：交易量多，價格還是低。<br>
<b>月份</b>（題目沒問，批改時要知道）：測試 Cost 降比較多（約 29 → 24），w<sub>1</sub> 也變成約 −12。季節對價格的影響，不是全部都透過交易量反映。學生放了月份，係數跟參考值不同是合理的。</li>
</ol>

</div>


In [ ]:
# 只用天氣齊全的日子，同一組訓練、測試切法，各加一次溫度、雨量、月份
weather_ok = (rows[:, 4] != "") & (rows[:, 5] != "")
temp = np.where(weather_ok, rows[:, 4], "0").astype(float)
rain = np.where(weather_ok, rows[:, 5], "0").astype(float)
tr_w = train[weather_ok[train]]
te_w = test[weather_ok[test]]
print("天氣齊全的日子：訓練", len(tr_w), "測試", len(te_w))

# 月份用 0/1 表示；1 月當基準，所以只放 2 到 12 月
month = rows[:, 1].astype(int)
month_cols = np.column_stack([(month == mo).astype(float) for mo in range(2, 13)])

def fit_and_test(X):
    Xn, m_, s_ = zscore_normalize_features(X[tr_w])
    w_n, b_n, _ = gradient_descent_multi(Xn, y_all[tr_w], alpha=0.1, num_iters=1000)
    w_o, b_o = to_original_scale(w_n, b_n, m_, s_)
    return w_o, compute_cost_multi((X[te_w] - m_) / s_, y_all[te_w], w_n, b_n)

for label, X in [
    ("ln(交易量)+年份", X_all),
    ("再加溫度", np.column_stack([X_all, temp])),
    ("再加雨量", np.column_stack([X_all, rain])),
    ("再加月份", np.column_stack([X_all, month_cols])),
]:
    w_o, c = fit_and_test(X)
    print(f"{label:14s} 測試 Cost {c:6.2f}  ln(交易量) 的 w {w_o[0]:7.2f}")
print("ln(交易量) 和溫度的相關係數", round(float(np.corrcoef(np.log(volume[weather_ok]), temp[weather_ok])[0, 1]), 2))
